# Module 9 · 文字 5：案例 — IMDB 情感分析（經典 vs 現代）

> **本節定位｜整合案例**
> 把 `01`（經典 TF-IDF）與 `02–03`（現代 tokenizer + 句向量）放進**同一個真實任務**，
> 直接比較：在 IMDB 影評情感分類上，**TF-IDF + 邏輯回歸** vs **句向量 + 邏輯回歸**。
> 全程 **CPU 可跑**（用小子集 + 輕量句向量模型）。

## 學習目標
- 建立一條可重現的文本分類流程：載入 → 表示 → 分類 → 評估。
- 量化比較「稀疏 TF-IDF」與「稠密句向量」兩種特徵的表現與維度差異。
- 體會句向量的優勢（低維、語意、可遷移），並理解「想再更好 → 端到端微調」是下一步 (M11)。

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

## 1. 載入資料（IMDB 小子集，CPU 友善）

優先用 HuggingFace `datasets` 載入 IMDB；若離線/未安裝，退回內建迷你樣本，確保 notebook 可跑。

In [ ]:
def load_imdb_subset(n_train=400, n_test=200):
    try:
        from datasets import load_dataset
        ds = load_dataset("imdb")
        tr = ds["train"].shuffle(seed=42).select(range(n_train))
        te = ds["test"].shuffle(seed=42).select(range(n_test))
        return (tr["text"], tr["label"]), (te["text"], te["label"]), "HuggingFace datasets: imdb"
    except Exception as e:
        # 離線後備：極小手造樣本（僅供流程跑通）
        pos = ["A brilliant and moving masterpiece.", "Absolutely loved every minute of it.",
               "Fantastic acting and a gripping story.", "One of the best films this year."]
        neg = ["A boring and predictable mess.", "I hated this dull, lifeless movie.",
               "Terrible script and wooden acting.", "Waste of time, utterly forgettable."]
        texts = pos*8 + neg*8
        labels = [1]*len(pos)*8 + [0]*len(neg)*8
        split = int(len(texts)*0.7)
        idx = np.random.RandomState(42).permutation(len(texts))
        tr_i, te_i = idx[:split], idx[split:]
        Xtr = [texts[i] for i in tr_i]; ytr = [labels[i] for i in tr_i]
        Xte = [texts[i] for i in te_i]; yte = [labels[i] for i in te_i]
        return (Xtr, ytr), (Xte, yte), "離線後備迷你樣本"

(X_train, y_train), (X_test, y_test), source = load_imdb_subset()
y_train, y_test = np.array(y_train), np.array(y_test)
print(f"資料來源：{source}")
print(f"訓練 {len(X_train)} 筆、測試 {len(X_test)} 筆（標籤：1=正面, 0=負面）")

## 2. 路線 A（經典）：TF-IDF + 邏輯回歸

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words="english")
Xtr_tfidf = tfidf.fit_transform(X_train)
Xte_tfidf = tfidf.transform(X_test)
print(f"TF-IDF 特徵維度 = {Xtr_tfidf.shape[1]}（稀疏）")

clf_a = LogisticRegression(max_iter=1000)
clf_a.fit(Xtr_tfidf, y_train)
pred_a = clf_a.predict(Xte_tfidf)
acc_a, f1_a = accuracy_score(y_test, pred_a), f1_score(y_test, pred_a)
print(f"[TF-IDF]   Accuracy = {acc_a:.3f} | F1 = {f1_a:.3f}")

## 3. 路線 B（現代）：句向量 + 邏輯回歸

用 `sentence-transformers` 的輕量模型把每篇影評編成 384 維稠密句向量，再用同樣的邏輯回歸分類。

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    Xtr_emb = st.encode(X_train, show_progress_bar=False)   # (N_train, 384)
    Xte_emb = st.encode(X_test, show_progress_bar=False)
    print(f"句向量特徵維度 = {Xtr_emb.shape[1]}（稠密，僅 384 維！）")

    clf_b = LogisticRegression(max_iter=1000)
    clf_b.fit(Xtr_emb, y_train)
    pred_b = clf_b.predict(Xte_emb)
    acc_b, f1_b = accuracy_score(y_test, pred_b), f1_score(y_test, pred_b)
    print(f"[句向量]   Accuracy = {acc_b:.3f} | F1 = {f1_b:.3f}")
except Exception as e:
    acc_b = f1_b = None
    print("（未安裝 sentence-transformers，略過路線 B）", e)

## 4. 比較與解讀

In [ ]:
print(f"{'方法':<14}{'特徵維度':<12}{'Accuracy':<10}{'F1':<8}")
print(f"{'TF-IDF':<14}{Xtr_tfidf.shape[1]:<12}{acc_a:<10.3f}{f1_a:<8.3f}")
if acc_b is not None:
    print(f"{'句向量(384)':<14}{Xtr_emb.shape[1]:<12}{acc_b:<10.3f}{f1_b:<8.3f}")

print("""
觀察重點：
- TF-IDF 用了數千維稀疏特徵；句向量只用 384 維稠密特徵就能達到相當（常更好）的表現。
- 句向量帶語意：對同義改寫、罕見措辭更穩健，且可直接遷移到檢索/分群等其他任務。
- 在小資料上兩者可能接近；資料越大、語言越自然，現代嵌入的優勢越明顯。
""")

## 小結與延伸

| 路線 | 特徵 | 維度 | 特性 |
|:--|:--|:--|:--|
| 經典 | TF-IDF | 數千（稀疏） | 強基線、可解釋，但上下文無關 |
| 現代 | 句向量 | 384（稠密） | 語意、低維、可遷移 |

**想要更高的天花板？** 以上是「**凍結特徵 + 傳統分類器**」。
若把 Transformer **端到端微調**（連同它的權重一起學），通常能再往上一截——
這正是 **Module 11 · `02_text_downstream`** 要做的：用這份 tokenized 資料微調一個
DistilBERT 分類器，並介紹 LLM 的 LoRA/SFT 微調。